<a href="https://colab.research.google.com/github/chadlomas2011-blip/Customer-Churn-AI-Analytics/blob/main/Construction_Industry_Lead_Loss_%26_Latency_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np

raw_data = {'lead_id': ['L-101', 'L-102', 'L-103', 'L-104', 'L-105', 'L-106', 'L-107'], 'timestamp_received': ['2026-08-20 09:15:00', '2026-08-20 10:00:00', '2026-08-20 11:30:00', '2026-08-20 14:10:00', '2026-08-21 08:05:00', '2026-08-21 09:00:00', '2026-08-21 16:45:00'], 'timestamp_first_contact': ['2026-08-20 09:18:00', None, '2026-08-20 16:45:00', '2026-08-20 14:11:00', None, '2026-08-22 10:00:00', '2026-08-21 16:48:00'], 'customer_name': [' John Smith ', 'SARAH CONNOR', 'Mike Johnson', 'spam_user', 'David Lee', 'Unknown', '  Emily Davis '], 'service_requested': ['roofing replacement', 'Deck Repair', 'ROOFING REPAIR', 'SEO Services', 'deck build', 'Roofing', 'gutter cleaning'], 'estimated_project_val': ['$12,000', '$4,500', '$2,800', '$0', '$15,000', '$8,500', 'INVALID'], 'status': ['Closed Won', 'Lost - No Answer', 'Lost - Went with Competitor', 'Spam', 'Lost - No Answer', 'Lost - Delayed Response', 'Closed Won']}

df = pd.DataFrame(raw_data)

df['customer_name'] = df['customer_name'].str.strip().str.title()
df['service_requested'] = df['service_requested'].str.strip().str.lower()
df['timestamp_received'] = pd.to_datetime(df['timestamp_received'])
df['timestamp_first_contact'] = pd.to_datetime(df['timestamp_first_contact'])
df['response_time_minutes'] = (df['timestamp_first_contact'] - df['timestamp_received']).dt.total_seconds() / 60.0

df['estimated_project_val'] = df['estimated_project_val'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['estimated_project_val'] = pd.to_numeric(df['estimated_project_val'], errors='coerce').fillna(0)

df_cleaned = df[df['status'] != 'Spam'].copy()

df_cleaned['response_bracket'] = pd.cut(df_cleaned['response_time_minutes'], bins=[-1, 5, 30, 240, 1440, np.inf], labels=['< 5 mins', '5-30 mins', '30m-4h', '4h-24h', 'No Response / > 24h']).fillna('No Response / > 24h')

df_cleaned['is_closed'] = (df_cleaned['status'] == 'Closed Won').astype(int)
df_cleaned['is_lost'] = df_cleaned['status'].str.startswith('Lost').astype(int)
df_cleaned['lost_rev_val'] = df_cleaned['is_lost'] * df_cleaned['estimated_project_val']

summary = df_cleaned.groupby('response_bracket', observed=False).agg(total_leads=('lead_id', 'count'), closed_deals=('is_closed', 'sum'), lost_deals=('is_lost', 'sum'), lost_revenue=('lost_rev_val', 'sum')).reset_index()

summary['conversion_rate'] = (summary['closed_deals'] / summary['total_leads']).fillna(0).map('{:.1%}'.format)

print(summary.to_string(index=False))


   response_bracket  total_leads  closed_deals  lost_deals  lost_revenue conversion_rate
           < 5 mins            2             2           0           0.0          100.0%
          5-30 mins            0             0           0           0.0            0.0%
             30m-4h            0             0           0           0.0            0.0%
             4h-24h            1             0           1        2800.0            0.0%
No Response / > 24h            3             0           3       28000.0            0.0%
